In [1]:
import pandas as pd
df = pd.read_parquet("../data/raw/crimes_2021_present.parquet")
df.shape, df.dtypes

((1393587, 21),
 id                       str
 case_number              str
 date                     str
 block                    str
 iucr                     str
 primary_type             str
 description              str
 location_description     str
 arrest                  bool
 domestic                bool
 beat                     str
 district                 str
 ward                     str
 community_area           str
 fbi_code                 str
 x_coordinate             str
 y_coordinate             str
 year                     str
 updated_on               str
 latitude                 str
 longitude                str
 dtype: object)

In [2]:
# Nulos por columna (en %)
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

longitude               1.47
latitude                1.47
y_coordinate            1.47
x_coordinate            1.47
location_description    0.52
community_area          0.01
district                0.00
updated_on              0.00
year                    0.00
fbi_code                0.00
ward                    0.00
id                      0.00
case_number             0.00
domestic                0.00
arrest                  0.00
description             0.00
primary_type            0.00
iucr                    0.00
block                   0.00
date                    0.00
beat                    0.00
dtype: float64

In [3]:
# Duplicados
df["id"].duplicated().sum(), df["case_number"].duplicated().sum()

(np.int64(0), np.int64(179))

In [4]:
# Fechas: recuerda que llegan como texto
df["date"] = pd.to_datetime(df["date"])
df["date"].min(), df["date"].max()

(Timestamp('2021-01-01 00:00:00'), Timestamp('2026-10-11 00:00:00'))

In [5]:
# Volumen mensual: ¿el último mes está incompleto?
df.set_index("date").resample("MS").size().tail(8)

date
2026-03-01    19191
2026-04-01    19166
2026-05-01    21163
2026-06-01    20068
2026-07-01    21291
2026-08-01    20844
2026-09-01    19747
2026-10-01     6345
Freq: MS, dtype: int64

In [6]:
# Coordenadas fuera de Chicago
lat = pd.to_numeric(df["latitude"]); lon = pd.to_numeric(df["longitude"])
fuera = ~lat.between(41.6, 42.1) | ~lon.between(-87.95, -87.5)
(fuera & lat.notna()).sum()

np.int64(2)

In [7]:
# Categorías
df["primary_type"].value_counts()

primary_type
THEFT                                308148
BATTERY                              248108
CRIMINAL DAMAGE                      157120
ASSAULT                              125394
MOTOR VEHICLE THEFT                  114508
DECEPTIVE PRACTICE                    95181
OTHER OFFENSE                         90525
BURGLARY                              50849
ROBBERY                               46587
WEAPONS VIOLATION                     43671
NARCOTICS                             34148
CRIMINAL TRESPASS                     26956
OFFENSE INVOLVING CHILDREN            10553
CRIMINAL SEXUAL ASSAULT               10150
SEX OFFENSE                            7717
PUBLIC PEACE VIOLATION                 5258
INTERFERENCE WITH PUBLIC OFFICER       3712
HOMICIDE                               3569
STALKING                               2971
ARSON                                  2616
CONCEALED CARRY LICENSE VIOLATION      1234
PROSTITUTION                           1191
LIQUOR LAW VIOLATIO

In [8]:
# Proporciones
df["arrest"].value_counts(normalize=True), df["domestic"].value_counts(normalize=True)

(arrest
 False    0.863996
 True     0.136004
 Name: proportion, dtype: float64,
 domestic
 False    0.807516
 True     0.192484
 Name: proportion, dtype: float64)

In [10]:
# Identificar registros con case_number duplicado
dup = (
    df[df["case_number"].duplicated(keep=False)]
    .sort_values(["case_number", "id"])
)

cols = [
    "case_number", "id", "date", "iucr",
    "primary_type", "description", "arrest", "block"
]

display(dup[cols].head(20))

# ¿Qué columnas cambian dentro de un mismo caso?
variaciones = (
    dup.groupby("case_number")
    .nunique()
    .gt(1)
    .mean()
    .round(2)
)

display(variaciones)

# Contar duplicados exactos ignorando id y updated_on
duplicados_exactos = df.drop(
    columns=["id", "updated_on"]
).duplicated().sum()

print("Duplicados exactos:", duplicados_exactos)

,case_number,id,date,iucr,primary_type,description,arrest,block
9,JE104664,25708,2021-01-06 05:10:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,008XX W 75TH ST
10,JE104664,25709,2021-01-06 07:56:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,008XX W 75TH ST
17,JE108294,25718,2021-01-09 20:55:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,002XX S SACRAMENTO BLVD
18,JE108294,25719,2021-01-09 21:07:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,002XX S SACRAMENTO BLVD
85,JE154771,25791,2021-02-28 05:05:00,0110,HOMICIDE,FIRST DEGREE MURDER,True,086XX S HERMITAGE AVE
86,JE154771,25792,2021-02-28 05:03:00,0110,HOMICIDE,FIRST DEGREE MURDER,True,086XX S HERMITAGE AVE
93,JE163081,25799,2021-03-08 18:25:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,000XX E 119TH ST
94,JE163081,25800,2021-03-08 18:17:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,000XX E 119TH ST
102,JE168486,25808,2021-03-14 04:55:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,067XX S SOUTH CHICAGO AVE
103,JE168486,25809,2021-03-14 04:55:00,0110,HOMICIDE,FIRST DEGREE MURDER,False,067XX S SOUTH CHICAGO AVE


id                      1.00
date                    0.80
block                   0.00
iucr                    0.00
primary_type            0.00
description             0.00
location_description    0.00
arrest                  0.02
domestic                0.01
beat                    0.00
district                0.00
ward                    0.00
community_area          0.00
fbi_code                0.00
x_coordinate            0.01
y_coordinate            0.01
year                    0.02
updated_on              0.23
latitude                0.01
longitude               0.01
dtype: float64

Duplicados exactos: 33


In [12]:
dup = df[df["case_number"].duplicated(keep=False)]
dup["primary_type"].value_counts()

primary_type
HOMICIDE    337
Name: count, dtype: int64

In [13]:
# Filas idénticas salvo id y updated_on (duplicados reales)
df.drop(columns=["id", "updated_on"]).duplicated().sum()

np.int64(33)